# Gemini Inference Manifest Generation

This notebook prepares Gemini's transcription manifest for evaluation. It merges raw transcription results with ground-truth segmentations.

**Runtime:** designed to run in a Jupyter kernel whose CWD is `model/colabs/` (so the sibling `common/` package is importable). The `notebook_docker` compose service at `model/notebook_docker/` provides that, with the required `loguru` and `google-cloud-storage` already installed. A stock `Open in Colab` badge would land in a kernel without either, so the badge was removed.

**Note:** This version is focused strictly on data merging and manifest generation. Analysis (WER calculation) and visualization are handled in a separate benchmark notebook.

It performs the following steps:

1.  **Loads existing ground truth** from the baseline manifest.
2.  **Maps Gemini segments** using the `batch_manifest.jsonl` to align raw API results with the benchmark offsets.
3.  **Generates a merged manifest** specifically containing the Gemini predictions.
4.  **Exports the final benchmark** back to Google Cloud Storage.

In [ ]:
import collections
import json
import re
from pathlib import Path
from typing import Any

from google.cloud import storage
from google.colab import auth
from loguru import logger

# @markdown ### GCP Configuration
GCP_PROJECT_ID = ""  # @param {type:"string"}
GCS_BUCKET = ""  # @param {type:"string"}
PROJECT_NAME = ""  # @param {type:"string"}

# Model Info
MODEL_ID = "gemini-3.1-flash-lite-preview"  # @param ["gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {type:"string"}
MODEL_VERSION = re.sub(r"[-\.]", "_", MODEL_ID)

EXPERIMENT_NAME = ""  # @param {type:"string"}

assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided and cannot be empty."
assert GCS_BUCKET, "GCS_BUCKET must be provided and cannot be empty."
assert PROJECT_NAME, "PROJECT_NAME must be provided and cannot be empty."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."

# Input/Output Paths
# Path where raw Gemini transcripts are stored - Updated to point to the valid batch results subfolder
GCS_INPUT_TRANSCRIPTS_DIR = (
    f"transcripts/{PROJECT_NAME}_audio/{MODEL_VERSION}/{EXPERIMENT_NAME}"
)
GCS_MAP_DIR = f"segmented_audio/{PROJECT_NAME}_audio"
GCS_MANIFEST_PATH = f"manifests/{PROJECT_NAME}_transcriptions.json"

# Path where the final merged inference manifest will be saved
GCS_FINAL_OUTPUT_DIR = f"inference_manifests/{PROJECT_NAME}_{MODEL_VERSION}"

# Filenames
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"
EXISTING_BENCHMARK_FILENAME = f"{PROJECT_NAME}_transcriptions.json"
UPDATED_BENCHMARK_FILENAME = f"{EXPERIMENT_NAME}.jsonl"

# Authenticate to GCS
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

from common.manifest import load_manifest

In [ ]:
# Global variable to store predictions for diagnostics
gemini_predictions = {}


def merge_gcs_results_to_manifest(
    baseline_data: list[dict[str, Any]],
    batch_manifest_data: list[dict[str, Any]],
    gcs_bucket_name: str,
    output_file: str,
    predictions_gcs_path: str,
) -> dict[str, Any]:
    global gemini_predictions
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(gcs_bucket_name)
    gemini_predictions.clear()

    blob = bucket.blob(predictions_gcs_path)

    # Fallback/Error logic for empty files
    if not blob.exists() or (blob.size is not None and blob.size == 0):
        logger.warning(
            f"Requested predictions file is empty or missing: gs://{gcs_bucket_name}/{predictions_gcs_path}"
        )
        # Search in subfolders
        prefix = f"{GCS_INPUT_TRANSCRIPTS_DIR}/batch_results/"
        blobs = list(bucket.list_blobs(prefix=prefix))
        valid_blobs = [
            b
            for b in blobs
            if b.name.endswith("predictions.jsonl") and (b.size or 0) > 0
        ]

        if valid_blobs:
            blob = valid_blobs[0]
            logger.info(
                f"Found valid alternative results at: gs://{gcs_bucket_name}/{blob.name}"
            )
        else:
            logger.error(
                f"CRITICAL: No valid (non-zero) predictions.jsonl found at root or in {prefix}"
            )
            return {
                "total": len(baseline_data),
                "matched": 0,
                "missing": len(baseline_data),
            }

    local_preds = "temp_predictions.jsonl"
    blob.download_to_filename(local_preds)

    with open(local_preds) as f:
        for line in f:
            if not line.strip():
                continue
            try:
                data = json.loads(line)
                file_uri = ""

                req_contents = data.get("request", {}).get("contents", [])
                req_parts = (
                    req_contents[0].get("parts", []) if req_contents else []
                )

                for p in req_parts:
                    if p.get("file_data"):
                        file_uri = p["file_data"]["file_uri"]
                        break
                if not file_uri:
                    continue

                filename = Path(file_uri).stem
                if "__seg" not in filename:
                    continue
                example_id, seg_key = filename.split("__seg")

                raw_text = (
                    data.get("response", {})
                    .get("candidates", [{}])[0]
                    .get("content", {})
                    .get("parts", [{}])[0]
                    .get("text", "")
                )

                # Since output is text/plain, we don't need to check for JSON string formatting
                parsed = raw_text.strip()

                gemini_predictions[(example_id, seg_key)] = parsed
            except Exception:
                continue

    # Map offsets to segment IDs
    offset_to_seg = collections.defaultdict(dict)
    for entry in batch_manifest_data:
        eid = entry.get("example_id", "")
        off = round(float(entry.get("offset", 0.0)), 3)
        offset_to_seg[eid][off] = entry.get("segment_id", "")

    merged_records = []
    matched_count = 0

    for b_info in baseline_data:
        gemini_text = ""
        example_id = Path(b_info["audio_filepath"]).stem
        b_offset = round(float(b_info.get("offset", 0.0)), 3)
        matched_seg_id = offset_to_seg.get(example_id, {}).get(b_offset)

        if (
            matched_seg_id
            and (example_id, matched_seg_id) in gemini_predictions
        ):
            gemini_text = gemini_predictions[(example_id, matched_seg_id)]
            matched_count += 1

        merged_records.append(
            {**b_info, f"pred_text_{MODEL_VERSION}": gemini_text}
        )

    if matched_count == 0:
        logger.error(
            f"MATCH FAILURE: 0 segments matched out of {len(baseline_data)}. Check ID alignment and rounding."
        )
    elif matched_count < len(baseline_data):
        logger.warning(
            f"Partial Match: Only {matched_count}/{len(baseline_data)} segments matched."
        )

    with open(output_file, "w", encoding="utf-8") as f_out:
        f_out.writelines(json.dumps(rec) + "\n" for rec in merged_records)

    return {
        "total": len(baseline_data),
        "matched": matched_count,
        "missing": len(baseline_data) - matched_count,
    }


def run_pipeline() -> None:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    PREDICTIONS_FILE = f"{GCS_INPUT_TRANSCRIPTS_DIR}/predictions.jsonl"

    bucket.blob(GCS_MANIFEST_PATH).download_to_filename(
        EXISTING_BENCHMARK_FILENAME
    )
    bucket.blob(
        f"{GCS_MAP_DIR}/{BATCH_MANIFEST_FILENAME}"
    ).download_to_filename(BATCH_MANIFEST_FILENAME)

    baseline_data = load_manifest(EXISTING_BENCHMARK_FILENAME)
    batch_manifest_data = load_manifest(BATCH_MANIFEST_FILENAME)

    stats = merge_gcs_results_to_manifest(
        baseline_data,
        batch_manifest_data,
        GCS_BUCKET,
        UPDATED_BENCHMARK_FILENAME,
        PREDICTIONS_FILE,
    )

    if stats["matched"] > 0:
        gcs_output_path = f"{GCS_FINAL_OUTPUT_DIR}/{UPDATED_BENCHMARK_FILENAME}"
        bucket.blob(gcs_output_path).upload_from_filename(
            UPDATED_BENCHMARK_FILENAME
        )
        logger.info(
            f"Successfully matched {stats['matched']} rows. Uploaded to gs://{GCS_BUCKET}/{gcs_output_path}"
        )

In [ ]:
# @title Run Processing
run_pipeline()